# POC Model Walkthrough

This notebook compares two versions of the same binary classifier using election data from **2012, 2017, and 2022**:

- `with_2022_t1`: uses 2017 T1, 2017 T2, 2022 T1, and security features
- `without_2022_t1`: removes all 2022 T1 election features

**Target:**
- `1` if Macron wins the commune in 2022 second round
- `0` otherwise

**Data Coverage:**
- 2012: T1 and T2 (6 candidates each round)
- 2017: T1 and T2 (11 candidates each round)
- 2022: T1 and T2 (12 candidates each round)
- 2012 special handling: Blancs et Nuls were aggregated by law, now decomposed with `blancs_et_nuls` column

In [ ]:
from __future__ import annotations

import csv
import io
import json
import math
import random
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1] if Path.cwd().name == 'ml' else Path.cwd().resolve()
while not (ROOT_DIR / 'data').exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent

ELECTION_DIR = ROOT_DIR / 'data' / 'gold' / 'election'
SECURITY_DIR = ROOT_DIR / 'data' / 'gold' / 'security'
OUTPUT_DIR = ROOT_DIR / 'src' / 'ml' / 'outputs'

print('ROOT_DIR =', ROOT_DIR)
print('ELECTION_DIR =', ELECTION_DIR)
print('SECURITY_DIR =', SECURITY_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
def read_csv_rows(path: Path) -> list[dict[str, str]]:
    content = path.read_text(encoding='utf-8')
    return list(csv.DictReader(io.StringIO(content), delimiter=';'))


def write_csv_rows(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def normalize_id_commune(value: str) -> str:
    parts = str(value).split('-', 1)
    if len(parts) == 2:
        dept = parts[0].strip().zfill(2)
        commune = parts[1].strip().replace('.0', '').zfill(3)
        return f'{dept}-{commune}'
    return str(value).strip()


def slugify(text: str) -> str:
    text = text.lower().strip()
    for old, new in [
        ('é', 'e'), ('è', 'e'), ('ê', 'e'), ('ë', 'e'),
        ('à', 'a'), ('â', 'a'), ('ä', 'a'),
        ('î', 'i'), ('ï', 'i'), ('ô', 'o'), ('ö', 'o'),
        ('ù', 'u'), ('û', 'u'), ('ü', 'u'), ('ç', 'c'),
        ("'", '_'), ('-', '_'), (' ', '_'), ('(', '_'), (')', '_'), ('/', '_')
    ]:
        text = text.replace(old, new)
    while '__' in text:
        text = text.replace('__', '_')
    return text.strip('_')


def to_float(value: str, default: float = 0.0) -> float:
    if value is None:
        return default
    raw = str(value).strip()
    if raw == '':
        return default
    try:
        return float(raw)
    except ValueError:
        return default


def to_int(value: str, default: int = 0) -> int:
    if value is None:
        return default
    raw = str(value).strip()
    if raw == '':
        return default
    try:
        return int(float(raw))
    except ValueError:
        return default


def sigmoid(x: float) -> float:
    x = max(-50.0, min(50.0, x))
    return 1.0 / (1.0 + math.exp(-x))


def dot_product(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


def std(values: list[float], avg: float) -> float:
    if not values:
        return 1.0
    variance = sum((v - avg) ** 2 for v in values) / len(values)
    return math.sqrt(variance) or 1.0


print('Basic helper functions loaded.')

In [ ]:
def classification_metrics(y_true: list[int], y_pred: list[int]) -> dict[str, float]:
    total = len(y_true)
    correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)
    accuracy = correct / total if total else 0.0

    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    balanced_accuracy = (recall + specificity) / 2.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }


def stratified_split(rows: list[dict[str, object]], target_key: str, test_size: float = 0.2, seed: int = 42):
    random.seed(seed)
    grouped = {0: [], 1: []}
    for row in rows:
        grouped[int(row[target_key])].append(row)

    train_rows = []
    test_rows = []

    for _, group in grouped.items():
        random.shuffle(group)
        split_idx = max(1, int(round(len(group) * (1 - test_size))))
        train_rows.extend(group[:split_idx])
        test_rows.extend(group[split_idx:])

    random.shuffle(train_rows)
    random.shuffle(test_rows)
    return train_rows, test_rows


def prepare_matrices(rows: list[dict[str, object]], feature_names: list[str]):
    matrix = []
    for row in rows:
        matrix.append([to_float(row[feature], default=float('nan')) for feature in feature_names])
    return matrix


def impute_and_scale(train_matrix: list[list[float]], test_matrix: list[list[float]]):
    n_features = len(train_matrix[0])
    means = []
    stds = []

    for j in range(n_features):
        values = [row[j] for row in train_matrix if not math.isnan(row[j])]
        avg = mean(values)
        s = std(values, avg)
        means.append(avg)
        stds.append(s)

    def transform(matrix: list[list[float]]) -> list[list[float]]:
        transformed = []
        for row in matrix:
            new_row = []
            for j, value in enumerate(row):
                v = means[j] if math.isnan(value) else value
                new_row.append((v - means[j]) / stds[j])
            transformed.append(new_row)
        return transformed

    return transform(train_matrix), transform(test_matrix)


def fit_logistic_regression(X: list[list[float]], y: list[int], lr: float = 0.05, epochs: int = 3000, l2: float = 0.001):
    n_features = len(X[0])
    weights = [0.0] * n_features
    bias = 0.0
    n_samples = len(X)

    for epoch in range(epochs):
        grad_w = [0.0] * n_features
        grad_b = 0.0

        for row, target in zip(X, y):
            pred = sigmoid(dot_product(row, weights) + bias)
            error = pred - target
            for j in range(n_features):
                grad_w[j] += error * row[j]
            grad_b += error

        for j in range(n_features):
            grad_w[j] = grad_w[j] / n_samples + l2 * weights[j]
            weights[j] -= lr * grad_w[j]
        bias -= lr * (grad_b / n_samples)

        if epoch in [0, 499, 999, 1999, 2999]:
            print(f'Epoch {epoch + 1}/{epochs} finished')

    return weights, bias


def predict_classes(X: list[list[float]], weights: list[float], bias: float):
    probas = [sigmoid(dot_product(row, weights) + bias) for row in X]
    preds = [1 if p >= 0.5 else 0 for p in probas]
    return preds, probas


print('Training helper functions loaded.')

In [ ]:
dim_election = read_csv_rows(ELECTION_DIR / 'dim_election.csv')
dim_candidat = read_csv_rows(ELECTION_DIR / 'dim_candidat.csv')
fact_resultats = read_csv_rows(ELECTION_DIR / 'fact_resultats_candidat.csv')
fact_participation = read_csv_rows(ELECTION_DIR / 'fact_participation.csv')
dim_nuance = read_csv_rows(ELECTION_DIR / 'dim_nuance.csv')
dim_indicateur = read_csv_rows(SECURITY_DIR / 'dim_indicateur_securite.csv')
fact_securite = read_csv_rows(SECURITY_DIR / 'fact_securite.csv')
fact_demographie = read_csv_rows(SECURITY_DIR / 'fact_demographie.csv')

print('dim_election rows =', len(dim_election))
print('dim_candidat rows =', len(dim_candidat))
print('fact_resultats_candidat rows =', len(fact_resultats))
print('fact_participation rows =', len(fact_participation))
print('dim_nuance rows =', len(dim_nuance))
print('dim_indicateur rows =', len(dim_indicateur))
print('fact_securite rows =', len(fact_securite))
print('fact_demographie rows =', len(fact_demographie))

if fact_resultats:
    print('Sample election row:', fact_resultats[0])
if fact_securite:
    print('Sample security row:', fact_securite[0])
if fact_demographie:
    print('Sample demographie row:', fact_demographie[0])

def build_security_dataset() -> dict[str, dict[str, float]]:
    """Agrège les données de sécurité et de démographie par commune (2022 et alentours)."""
    dataset = {}
    
    # 1. Démographie
    for row in fact_demographie:
        if row['annee'] == '2022':  # Ou la dernière année dispo
            id_commune = normalize_id_commune(row['id_commune'])
            feat = dataset.setdefault(id_commune, {})
            feat['insee_pop'] = to_float(row['insee_pop'])
            feat['insee_log'] = to_float(row['insee_log'])
            
    # 2. Sécurité
    indicateur_map = {r['id_indicateur_securite']: slugify(r['indicateur']) for r in dim_indicateur}
    for row in fact_securite:
        # On peut cibler une année ou faire la moyenne
        if row['annee'] == '2022':
            id_commune = normalize_id_commune(row['id_commune'])
            ind_name = indicateur_map.get(row['id_indicateur_securite'], 'inconnu')
            feat = dataset.setdefault(id_commune, {})
            feat[f'sec_rate_{ind_name}'] = to_float(row['taux_pour_mille'])
            feat[f'sec_count_{ind_name}'] = to_float(row['nombre'])

    return dataset

print('Data loaded and build_security_dataset function injected.')


In [ ]:
# ✅ VERIFY: Show available election years and tours
print("\n" + "="*80)
print("📊 AVAILABLE ELECTION DATA IN GOLD LAYER")
print("="*80)

election_summary = {}
for row in dim_election:
    year = row['annee_election']
    tour = row['tour']
    key = f"{year}_T{tour}"
    election_summary[key] = row['id_election']

print(f"\nTotal elections: {len(election_summary)}")
for key in sorted(election_summary.keys()):
    print(f"  ✓ {key}")

# Verify blancs_et_nuls column exists (new for 2012 harmonization)
sample_participation = fact_participation[0] if fact_participation else {}
if 'blancs_et_nuls' in sample_participation or 'blancs_et_nuls_ins' in sample_participation or 'blancs_et_nuls_vot' in sample_participation:
    print("\n✅ Blancs et Nuls column present (2012 harmonization applied)")
else:
    print("\n⚠️  Blancs et Nuls column not found - check data load")

# Summary statistics
communes_in_fact = set(normalize_id_commune(r['id_commune']) for r in fact_resultats)
print(f"\nTotal communes with results: {len(communes_in_fact)}")
print(f"Total result rows (candidate-commune-election): {len(fact_resultats)}")

print("="*80 + "\n")

In [ ]:
def run_experiment(experiment_name: str, include_2022_t1: bool = True):
    print('\n' + '=' * 80)
    print(f'EXPERIMENT = {experiment_name} | INCLUDE 2022 T1 = {include_2022_t1}')
    print('='*80 + f' | DATA: 2012, 2017, 2022 | COMMUNES: {len(communes_in_fact)}')
    print('=' * 80)

    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    candidate_map = {row['id_candidat']: row['nom'] for row in dim_candidat}

    feature_data = {}
    macron_scores_2022_t2 = {}

    # 1. Démographie & Participation (now includes blancs_et_nuls harmonization for 2012)
    for row in fact_participation:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        commune_features = feature_data.setdefault(id_commune, {})
        
        if not include_2022_t1 and election_key == '2022_T1':
            continue
        
        # Extended metrics including blancs_et_nuls for consistent 2012-2017-2022 comparison
        participation_metrics = [
            'inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 
            'blancs_et_nuls',  # ✨ New: Harmonizes 2012 aggregated data
            'exprimes', 
            'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins',
            'pct_blancs_et_nuls_ins', 'pct_blancs_et_nuls_vot',  # ✨ New: 2012 harmonization metrics
            'pct_exprimes_ins', 'pct_exprimes_vot'
        ]
        
        for metric in participation_metrics:
            if metric in row:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    # 2. Résultats des candidats (Agnostiques au besoin, mais ici on vise Macron)
    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        candidat_nom = candidate_map[row['id_candidat']].lower()
        commune_features = feature_data.setdefault(id_commune, {})

        if election_key == '2022_T2':
            if 'macron' in candidat_nom:
                macron_scores_2022_t2[id_commune] = to_float(row['pct_voix_exprimes'])
        else:
            if not include_2022_t1 and election_key == '2022_T1':
                continue
            
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(row.get('id_nuance', 'inconnu'))}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix

    # 3. Y value = 1 si Macron gagne (score > 50%)
    targets = {id_commune: 1 if score > 50.0 else 0 for id_commune, score in macron_scores_2022_t2.items()}
    security_features = build_security_dataset()

    all_feature_names = set()
    dataset_rows = []

    for id_commune, target_y in targets.items():
        r = {'id_commune': id_commune, 'target_macron_wins_2022_t2': target_y}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        all_feature_names.update(k for k in r.keys() if k not in {'id_commune', 'target_macron_wins_2022_t2'})

    feature_names = sorted(all_feature_names)
    for r in dataset_rows:
        for f in feature_names:
            r.setdefault(f, 0.0 if 'share_nuance' in f else None)

    print('Dataset rows =', len(dataset_rows))
    print('Feature count =', len(feature_names))

    train_rows, test_rows = stratified_split(dataset_rows, 'target_macron_wins_2022_t2', test_size=0.2, seed=42)
    train_matrix = prepare_matrices(train_rows, feature_names)
    test_matrix = prepare_matrices(test_rows, feature_names)
    train_matrix, test_matrix = impute_and_scale(train_matrix, test_matrix)

    y_train = [int(r['target_macron_wins_2022_t2']) for r in train_rows]
    y_test = [int(r['target_macron_wins_2022_t2']) for r in test_rows]

    weights, bias = fit_logistic_regression(train_matrix, y_train)
    pred_classes, pred_probas = predict_classes(test_matrix, weights, bias)

    model_metrics = classification_metrics(y_test, pred_classes)

    feature_importance = []
    for feature, weight in sorted(zip(feature_names, weights), key=lambda item: abs(item[1]), reverse=True):
        feature_importance.append({
            'feature': feature,
            'coefficient': weight,
            'abs_coefficient': abs(weight),
        })

    return {
        'features_count': len(feature_names),
        'model_metrics': model_metrics,
        'feature_importance': feature_importance
    }

print('✅ run_experiment defined for Macron classification (2012, 2017, 2022 data included).')

In [ ]:
result_with_2022_t1 = run_experiment('with_2022_t1', include_2022_t1=True)
result_without_2022_t1 = run_experiment('without_2022_t1', include_2022_t1=False)           

In [ ]:
comparison = {
    'metadata': {
        'description': 'Binary classification: Macron wins 2022 T2 (target=1) vs does not win (target=0)',
        'data_coverage': ['2012 (T1/T2)', '2017 (T1/T2)', '2022 (T1/T2)'],
        'communes_analyzed': len(communes_in_fact),
        'features_included': ['participation metrics', 'candidate results (nuances)', 'security indicators'],
        'note_2012': '2012 data has blancs/nuls aggregated by law - now decomposed with blancs_et_nuls column'
    },
    'with_2022_t1': {
        'features_count': result_with_2022_t1['features_count'],
        'accuracy': result_with_2022_t1['model_metrics']['accuracy'],
        'balanced_accuracy': result_with_2022_t1['model_metrics']['balanced_accuracy'],
        'f1': result_with_2022_t1['model_metrics']['f1'],
    },
    'without_2022_t1': {
        'features_count': result_without_2022_t1['features_count'],
        'accuracy': result_without_2022_t1['model_metrics']['accuracy'],
        'balanced_accuracy': result_without_2022_t1['model_metrics']['balanced_accuracy'],
        'f1': result_without_2022_t1['model_metrics']['f1'],
    },
}

print('COMPARISON SUMMARY (2012, 2017, 2022)')
print('='*80)
print(json.dumps(comparison, indent=2))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
print('\n✅ Comparison file saved to =', OUTPUT_DIR / 'comparison.json')

In [ ]:
print('TOP 10 FEATURES WITH 2022 T1')
for row in result_with_2022_t1['feature_importance'][:10]:
    print(row)

print('\nTOP 10 FEATURES WITHOUT 2022 T1')
for row in result_without_2022_t1['feature_importance'][:10]:
    print(row)

## Interpretation

- If the model without `2022 T1` still performs well, the POC is stronger because it is less dependent on the immediately previous round.
- The main file to cite in your report is `comparison.json`.
- The detailed outputs for each experiment are stored in:
  - `src/ml/outputs/with_2022_t1`
  - `src/ml/outputs/without_2022_t1`

## Nouvelle approche : Prédictions généralisées par Nuance Politique

L'objectif de cette cellule est de créer un **nouveau modèle** qui prédit la probabilité de victoire d'une **Nuance Politique spécifique** au second tour (ex: `REM` ou `RN`), en se basant *uniquement* sur les scores passés des nuances (et sans utiliser l'historique des noms des candidats).

Cela permet d'avoir un algorithme "agnostique" des personnes, qui pourrait s'appliquer théoriquement à d'autres contextes territoriaux en se basant uniquement sur la polarité politique.

In [ ]:
def run_nuance_centric_model(target_nuance: str, experiment_name: str, include_2022_t1: bool = True):
    print('\n' + '=' * 80)
    print(f'NEW EXPERIMENT = {experiment_name} | TARGET NUANCE = {target_nuance}')
    print('=' * 80)

    # 1. Extraction mapping
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    candidate_map = {row['id_candidat']: {'nom': row['nom']} for row in dim_candidat} # id_nuance est mnt dans fact_resultats

    feature_data = {}
    target_scores = {}

    # 2. Construction des features démographiques liées à l'élection (Issue de fact_participation)
    for row in fact_participation:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        commune_features = feature_data.setdefault(id_commune, {})
        
        if not include_2022_t1 and election_key == '2022_T1':
            continue
            
        for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
            if metric in row:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    # 3. Construction des features résultats (Issue de fact_resultats_candidat)
    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        id_nuance_actuelle = row.get('id_nuance', 'inconnu')
        commune_features = feature_data.setdefault(id_commune, {})

        if election_key == '2022_T2':
            # On cherche qui a gagné le T2 et sa NUANCE
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, id_nuance_actuelle)
        else:
            if not include_2022_t1 and election_key == '2022_T1':
                continue
            
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(id_nuance_actuelle)}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix

    # 4. Y value = 1 si la nuance ciblée gagne
    targets = {id_commune: 1 if winner_nuance == target_nuance else 0 for id_commune, (_, winner_nuance) in target_scores.items()}
    security_features = build_security_dataset()

    # 5. Préparation Matrice
    all_feature_names = set()
    dataset_rows = []
    target_col = f'target_{target_nuance}_wins_2022_t2'

    for id_commune, target_y in targets.items():
        r = {'id_commune': id_commune, target_col: target_y}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        all_feature_names.update(k for k in r.keys() if k not in {'id_commune', target_col})

    feature_names = sorted(all_feature_names)
    for r in dataset_rows:
        for f in feature_names:
            r.setdefault(f, None)

    print('Dataset rows =', len(dataset_rows))
    print('Feature count =', len(feature_names))
    
    # Check what features look like
    print('Examples of features used:', [f for f in feature_names if 'share' in f][:5])

    train_rows, test_rows = stratified_split(dataset_rows, target_col, test_size=0.2, seed=42)
    train_matrix = prepare_matrices(train_rows, feature_names)
    test_matrix = prepare_matrices(test_rows, feature_names)
    train_matrix, test_matrix = impute_and_scale(train_matrix, test_matrix)

    y_train = [int(r[target_col]) for r in train_rows]
    y_test = [int(r[target_col]) for r in test_rows]

    weights, bias = fit_logistic_regression(train_matrix, y_train)
    pred_classes, pred_probas = predict_classes(test_matrix, weights, bias)

    model_metrics = classification_metrics(y_test, pred_classes)
    print('Model metrics =')
    print(json.dumps(model_metrics, indent=2))

    # 6. Export des résultats
    predictions = []
    for r, pred, proba in zip(test_rows, pred_classes, pred_probas):
        predictions.append({
            'id_commune': r['id_commune'],
            target_col: r[target_col],
            'predicted_class': pred,
            f'predicted_proba_{target_nuance}': proba,
        })

    feature_importance = []
    for feature, weight in sorted(zip(feature_names, weights), key=lambda item: abs(item[1]), reverse=True):
        feature_importance.append({
            'feature': feature,
            'coefficient': weight,
            'abs_coefficient': abs(weight),
        })

    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)

    write_csv_rows(experiment_dir / 'model_dataset.csv', dataset_rows, ['id_commune', target_col] + feature_names)
    write_csv_rows(experiment_dir / 'test_predictions.csv', predictions, ['id_commune', target_col, 'predicted_class', f'predicted_proba_{target_nuance}'])
    write_csv_rows(experiment_dir / 'feature_importance.csv', feature_importance, ['feature', 'coefficient', 'abs_coefficient'])

    print(f'Done! Predictions saved to: {experiment_dir / "test_predictions.csv"}')
    return feature_importance, predictions

print('Nuance model function loaded.')

In [ ]:
# Modèle ciblant la prédiction de la victoire de la nuance REM (majorité présidentielle)
feat_imp_rem, preds_rem = run_nuance_centric_model(
    target_nuance='REM', 
    experiment_name='nuance_predict_winner_REM', 
    include_2022_t1=True
)

print('\nTOP 10 VARIABLES Poussant à la victoire de la nuance REM :')
for row in feat_imp_rem[:10]:
    print(row)


## Modèle Multi-Classes (Prédiction de la nuance gagnante)

Plutôt que de faire une classification binaire (Vrai ou Faux pour une nuance spécifique), nous entraînons ici un modèle **Multi-Classes** avec l'approche "One-vs-Rest". L'algorithme calcule indépendamment la probabilité de victoire de chaque nuance, et prédit celle qui obtient le score de confiance le plus élevé. On a donc directement la nuance vainqueur pour chaque commune.

In [ ]:
def run_multiclass_nuance_model(experiment_name: str, include_2022_t1: bool = True):
    print('\n' + '=' * 80)
    print(f'MULTICLASS EXPERIMENT = {experiment_name} | INCLUDE 2022 T1 = {include_2022_t1}')
    print('=' * 80)

    # 1. Extraction mapping
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}

    feature_data = {}
    target_scores = {}

    # 2. Construction des features demographiques
    for row in fact_participation:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        commune_features = feature_data.setdefault(id_commune, {})
        
        if not include_2022_t1 and election_key == '2022_T1':
            continue
            
        for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
            if metric in row:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    # 3. Construction des features pour chaque candidat (Agnostique)
    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        id_nuance_actuelle = row.get('id_nuance', 'inconnu')
        commune_features = feature_data.setdefault(id_commune, {})

        # On cible l'élection 2022 T2 pour déterminer la nuance gagnante finale
        if election_key == '2022_T2':
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, id_nuance_actuelle)
        else:
            # STOP DATA LEAKAGE: Ignore 2022_T1 if requested
            if not include_2022_t1 and election_key == '2022_T1':
                continue
                
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(id_nuance_actuelle)}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix

    # 3b. Y value = La classe gagnante (ex: 'REM', 'RN')
    targets_nuance = {id_commune: nuance for id_commune, (_, nuance) in target_scores.items()}
    unique_classes = sorted(list(set(targets_nuance.values())))

    security_features = build_security_dataset()

    # 4. Préparation Matrice
    all_feature_names = set()
    dataset_rows = []

    for id_commune, actual_nuance in targets_nuance.items():
        r = {'id_commune': id_commune, 'target_nuance': actual_nuance}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        for k in r.keys():
             if k not in {'id_commune', 'target_nuance'}:
                 all_feature_names.add(k)

    feature_names = sorted(all_feature_names)
    for r in dataset_rows:
        for f in feature_names:
            if 'share_nuance' in f:
                r.setdefault(f, 0.0)
            else:
                r.setdefault(f, None)
                
    # Fix Split stratifié pour des keys en chaine de caractères
    random.seed(42)
    train_rows = []
    test_rows = []
    grouped = {}
    for r in dataset_rows:
        grouped.setdefault(r['target_nuance'], []).append(r)
    
    for cls, group in grouped.items():
        random.shuffle(group)
        split_idx = max(1, int(round(len(group) * 0.8))) 
        train_rows.extend(group[:split_idx])
        test_rows.extend(group[split_idx:])

    random.shuffle(train_rows)
    random.shuffle(test_rows)

    train_matrix = prepare_matrices(train_rows, feature_names)
    test_matrix = prepare_matrices(test_rows, feature_names)
    train_matrix, test_matrix = impute_and_scale(train_matrix, test_matrix)

    # 5. Entraînement "One-vs-Rest"
    models = {}
    for cls in unique_classes:
        y_train_binary = [1 if r['target_nuance'] == cls else 0 for r in train_rows]
        weights, bias = fit_logistic_regression(train_matrix, y_train_binary, epochs=1500)
        models[cls] = {'weights': weights, 'bias': bias}

    # 6. Prédiction
    predictions = []
    correct = 0
    
    for i, row in enumerate(test_rows):
        best_cls = None
        best_proba = -1
        
        for cls in unique_classes:
            w = models[cls]['weights']
            b = models[cls]['bias']
            p = sigmoid(dot_product(test_matrix[i], w) + b)
            if p > best_proba:
                best_proba = p
                best_cls = cls
                
        if best_cls == row['target_nuance']:
            correct += 1
            
    accuracy = correct / len(test_rows) if test_rows else 0
    print(f"\n✅ Multi-class Accuracy : {accuracy * 100:.2f} %")
    return predictions, accuracy

print('Fonction Multi-Classes réparée pour strings et split facts.')

In [ ]:
# RUN 1: Avec les données du 1er tour de 2022 (Risque de temporal data leakage)
_, acc_with_t1 = run_multiclass_nuance_model('multiclass_with_2022_t1', include_2022_t1=True)

# RUN 2: Sans les données du 1er tour de 2022 (Prédiction sur le long terme)
_, acc_without_t1 = run_multiclass_nuance_model('multiclass_without_2022_t1', include_2022_t1=False)

print(f"\n--- CONCLUSION ---")
print(f"Accuracy avec le 1er tour juste avant l'élection: {acc_with_t1*100:.2f}%")
print(f"Accuracy sans le 1er tour (seulement 2017 + Sécurité): {acc_without_t1*100:.2f}%")

## Un modèle beaucoup plus "Intelligent" : Random Forest (Forêts Aléatoires)

La régression logistique linéaire que nous avons construite de zéro donne de bons résultats, mais elle ne peut pas comprendre les interactions complexes entre les variables (ex: effet combiné du taux d'abstention ET de l'insécurité).

Pour surpasser ses performances, nous utilisons ici **Scikit-Learn** et un **Random Forest Classifier**.  
- Il génère des centaines d'arbres de décision qui votent ensemble.
- Il est très résistant à l'Overfitting (grâce au paramètre `max_depth` et au *Bagging*).
- Il nous donne une "Feature Importance" beaucoup plus fidèle à la réalité car non impactée par la colinéarité des variables.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

def run_smarter_rf_model(experiment_name: str, include_2022_t1: bool = True):
    print('\n' + '=' * 80)
    print(f'SMARTER MULTICLASS MODEL: RANDOM FOREST | INCLUDE 2022 T1 = {include_2022_t1}')
    print('=' * 80)

    # 1. Pipeline de récupération des données agnostiques (Nuances unqiuement)
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    nuance_libelles = {row['id_nuance']: row['libelle_nuance'] for row in dim_nuance}

    feature_data = {}
    target_scores = {}

    # Démographie
    for row in fact_participation:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        commune_features = feature_data.setdefault(id_commune, {})
        
        if not include_2022_t1 and election_key == '2022_T1':
            continue
            
        for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
            if metric in row:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    # Résultats candidats
    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        id_nuance_actuelle = row.get('id_nuance', 'inconnu')
        commune_features = feature_data.setdefault(id_commune, {})

        if election_key == '2022_T2':
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            # On veut prédire qui (quelle nuance) gagne le T2
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, id_nuance_actuelle)
        else:
            if not include_2022_t1 and election_key == '2022_T1':
                continue
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(id_nuance_actuelle)}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix
            
    targets_nuance = {id_commune: nuance for id_commune, (_, nuance) in target_scores.items()}
    security_features = build_security_dataset()

    # 2. Construction d'un Dataset tabulaire pour Pandas
    dataset_rows = []
    all_features = set()
    for id_commune, actual_nuance in targets_nuance.items():
        r = {'id_commune': id_commune, 'target_nuance': actual_nuance}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        for k in r.keys():
             if k not in {'id_commune', 'target_nuance'}:
                 all_features.add(k)

    df = pd.DataFrame(dataset_rows)
    feature_cols = sorted(list(all_features))
    
    # 3. Nettoyage intelligent des valeurs nulles
    for f in feature_cols:
        if 'share_nuance' in f:
            df[f] = df[f].fillna(0.0) # Un parti absent = 0 votes
        elif 'sec_rate_' in f:
            df[f] = df[f].fillna(0.0) # Pas de criminalité recensée = 0%
        elif 'sec_count_' in f:
            df[f] = df[f].fillna(0.0)
        else:
            df[f] = df[f].fillna(df[f].median()) # Statistiques manquantes (population) = Médiane locale

    print(f"Dataset shape : {df.shape}")

    # 4. Split Train / Test
    X = df[feature_cols]
    y = df['target_nuance']
    
    X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
        X, y, df['id_commune'], test_size=0.2, random_state=42, stratify=y
    )

    # 5. Entraînement du modèle (100 Arbres formés à profondeur maximale de 15)
    rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)

    # 6. Evaluation du modèle
    preds = rf.predict(X_test)
    probas = rf.predict_proba(X_test)
    classes = rf.classes_
    acc = accuracy_score(y_test, preds)
    
    print(f"\n✅ Random Forest Test Accuracy : {acc * 100:.2f} %")
    print("\n--- RAPPORT PAR NUANCE (Classification Report) ---")
    noms_classes = [nuance_libelles.get(c, c) for c in classes]
    print(classification_report(y_test, preds, target_names=noms_classes, zero_division=0))

    # 7. Importance réelle des variables
    importances = rf.feature_importances_
    df_importances = pd.DataFrame({'Variable': feature_cols, 'Importance': importances})
    df_importances = df_importances.sort_values('Importance', ascending=False)
    
    print("\n🌲 TOP 10 DES VARIABLES LES PLUS DÉTECTÉES PAR LA FORÊT :")
    display(df_importances.head(10))
    
    # 8. Création du Dataframe de résultats (Test Set uniquement)
    results_test = pd.DataFrame({
        'id_commune': indices_test,
        'actual_winner': y_test,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y_test],
        'predicted_winner': preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in preds]
    })
    for i, c in enumerate(classes):
        results_test[f'proba_{c}'] = probas[:, i]

    # 8b. Création du Dataframe de résultats (Toutes les communes)
    all_preds = rf.predict(X)
    all_probas = rf.predict_proba(X)
    results_all = pd.DataFrame({
        'id_commune': df['id_commune'],
        'actual_winner': y,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y],
        'predicted_winner': all_preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in all_preds]
    })
    for i, c in enumerate(classes):
        results_all[f'proba_{c}'] = all_probas[:, i]
        
    # 9. Sauvegarde des résultats
    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    
    df.to_csv(experiment_dir / 'model_dataset.csv', index=False)
    results_test.to_csv(experiment_dir / 'test_predictions.csv', index=False)
    results_all.to_csv(experiment_dir / 'all_predictions.csv', index=False)
    df_importances.to_csv(experiment_dir / 'feature_importance.csv', index=False)
    
    print(f"\n📂 Résultats sauvegardés dans : {experiment_dir}")
        
    return rf, df_importances, results_all

# --- Exécution d'un "VRAI" test difficile (Sans les tendances du 1er tour de 2022) ---
rf_model, imp_df, all_predictions_df = run_smarter_rf_model(experiment_name='random_forest_without_t1', include_2022_t1=False)

## Test Ultime : Prédire le Premier Tour (2022 T1)

Le premier tour est beaucoup plus complexe car il y a une multitude de candidats et de nuances (LFI, LR, RN, REM, REC...). Ce modèle va tenter de deviner quelle nuance arrivera en tête dans chaque commune lors du **1er tour de 2022**, en s'appuyant *uniquement* sur les données historiques de 2017 et l'insécurité.

In [ ]:
def run_rf_model_for_2022_t1(experiment_name: str):
    print('\n' + '=' * 80)
    print(f'PREDICTING 2022 ROUND 1 (T1) WINNER: RANDOM FOREST')
    print('=' * 80)

    # 1. Pipeline de récupération
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    nuance_libelles = {row['id_nuance']: row['libelle_nuance'] for row in dim_nuance}

    feature_data = {}
    target_scores = {}

    # Démographie
    for row in fact_participation:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        commune_features = feature_data.setdefault(id_commune, {})
        
        # Uniquement les données de 2017 (T1 et T2) pour éviter le leakage !
        if election_key.startswith('2017'):
            for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
                if metric in row:
                    commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    # Résultats candidats
    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        id_nuance_actuelle = row.get('id_nuance', 'inconnu')
        commune_features = feature_data.setdefault(id_commune, {})

        # Cible = On cherche qui a gagné le T1 de 2022
        if election_key == '2022_T1':
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, id_nuance_actuelle)
        
        # Features = Uniquement les données de 2017 (T1 et T2) pour éviter le leakage !
        elif election_key.startswith('2017'):
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(id_nuance_actuelle)}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix

    targets_nuance = {id_commune: nuance for id_commune, (_, nuance) in target_scores.items()}
    security_features = build_security_dataset()

    # 2. Construction d'un Dataset tabulaire pour Pandas
    dataset_rows = []
    all_features = set()
    for id_commune, actual_nuance in targets_nuance.items():
        r = {'id_commune': id_commune, 'target_nuance': actual_nuance}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        for k in r.keys():
             if k not in {'id_commune', 'target_nuance'}:
                 all_features.add(k)

    df = pd.DataFrame(dataset_rows)
    feature_cols = sorted(list(all_features))
    
    # 3. Nettoyage intelligent des valeurs nulles
    for f in feature_cols:
        if 'share_nuance' in f:
            df[f] = df[f].fillna(0.0)
        elif 'sec_rate_' in f:
            df[f] = df[f].fillna(0.0)
        elif 'sec_count_' in f:
            df[f] = df[f].fillna(0.0)
        else:
            df[f] = df[f].fillna(df[f].median())

    # Suppression des classes ultra-minoritaires (1 seul représentant) 
    # car elles empêchent Sklearn de faire un "Stratify" propre lors de la division Train/Test
    class_counts = df['target_nuance'].value_counts()
    valid_classes = class_counts[class_counts > 1].index
    if len(valid_classes) < len(class_counts):
        print(f"⚠️ Retrait des nuances ayant gagné dans une seule commune (trop rare) : {set(class_counts.index) - set(valid_classes)}")
        df = df[df['target_nuance'].isin(valid_classes)]

    print(f"Dataset shape : {df.shape}")
    print(f"Nuances possibles à prédire : {df['target_nuance'].unique()}")

    # 4. Split Train / Test
    X = df[feature_cols]
    y = df['target_nuance']
    
    X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
        X, y, df['id_commune'], test_size=0.2, random_state=42, stratify=y
    )

    # 5. Entraînement du modèle
    rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)

    # 6. Evaluation du modèle
    preds = rf.predict(X_test)
    probas = rf.predict_proba(X_test)
    classes = rf.classes_
    acc = accuracy_score(y_test, preds)
    
    print(f"\n✅ Random Forest Test Accuracy (2022 T1) : {acc * 100:.2f} %")
    print("\n--- RAPPORT PAR NUANCE (Classification Report) ---")
    noms_classes = [nuance_libelles.get(c, c) for c in classes]
    print(classification_report(y_test, preds, target_names=noms_classes, zero_division=0))

    # 7. Importance réelle des variables
    importances = rf.feature_importances_
    df_importances = pd.DataFrame({'Variable': feature_cols, 'Importance': importances})
    df_importances = df_importances.sort_values('Importance', ascending=False)
    
    print("\n🌲 TOP 10 DES VARIABLES LES PLUS DÉTECTÉES PAR LA FORÊT :")
    display(df_importances.head(10))
    
    # 8. Création des Dataframes de résultats
    results_test = pd.DataFrame({
        'id_commune': indices_test,
        'actual_winner': y_test,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y_test],
        'predicted_winner': preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in preds]
    })
    for i, c in enumerate(classes):
        results_test[f'proba_{c}'] = probas[:, i]

    all_preds = rf.predict(X)
    all_probas = rf.predict_proba(X)
    results_all = pd.DataFrame({
        'id_commune': df['id_commune'],
        'actual_winner': y,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y],
        'predicted_winner': all_preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in all_preds]
    })
    for i, c in enumerate(classes):
        results_all[f'proba_{c}'] = all_probas[:, i]
        
    # 9. Sauvegarde des résultats
    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    
    df.to_csv(experiment_dir / 'model_dataset.csv', index=False)
    results_test.to_csv(experiment_dir / 'test_predictions.csv', index=False)
    results_all.to_csv(experiment_dir / 'all_predictions.csv', index=False)
    df_importances.to_csv(experiment_dir / 'feature_importance.csv', index=False)
    
    print(f"\n📂 Résultats T1 sauvegardés dans : {experiment_dir}")
        
    return rf, df_importances, results_all

# --- Exécution pour le Premier Tour 2022 ---
rf_t1_model, imp_t1_df, all_t1_predictions_df = run_rf_model_for_2022_t1(experiment_name='random_forest_2022_t1')

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option('display.max_rows', 50)

print('\n' + '=' * 80)
print('🔎 VISUALISATION : QUELLE NUANCE VA GAGNER DANS CHAQUE COMMUNE ?')
print('=' * 80)

# 1. Charger les noms des communes depuis la couche Gold
df_communes = pd.read_csv(ELECTION_DIR / 'dim_commune.csv', sep=';')

# 2. L'ID des communes est désormais au format INSEE pur sans tirets ('69001'), on le récupère tel quel
df_communes['id_commune_norm'] = df_communes['id_commune'].apply(normalize_id_commune)

# 3. Fusionner les prédictions (ex: all_t1_predictions_df) avec le nom de la commune
df_display = pd.merge(all_t1_predictions_df, df_communes, left_on='id_commune', right_on='id_commune_norm', how='left', suffixes=('', '_drop'))

# Ajout du Score de Confiance (Probabilité associée au vainqueur prédit)
# On récupère dynamiquement la valeur dans la colonne "proba_XYZ" selon la victoire prédite
df_display['confidence_score'] = df_display.apply(lambda row: row[f"proba_{row['predicted_winner']}"], axis=1)

# 4. Sélectionner et renommer les colonnes pour une lecture facile
visu_df = df_display[['id_commune', 'libelle_commune', 'predicted_winner', 'predicted_winner_label', 'confidence_score']].copy()

visu_df = visu_df.rename(columns={
    'id_commune': 'Code Insee',
    'libelle_commune': 'Commune',
    'predicted_winner': 'Sigle Prédit',
    'predicted_winner_label': '🏆 Nuance Politique Gagnante Prédite (Libellé)',
    'confidence_score': 'Score de Confiance'
})

# Mise en forme du pourcentage
visu_df['Score de Confiance'] = (visu_df['Score de Confiance'] * 100).round(2).astype(str) + ' %'

# 5. Afficher le tableau complet dans l'interface
display(visu_df)

In [ ]:
# 📊 SUMMARY: Data Expansion Impact
print("\n" + "="*80)
print("📈 DATA EXPANSION SUMMARY")
print("="*80)

print(f"""
✅ EXPANDED DATASET NOW INCLUDES:

Election Years:
  • 2012: Presidential election T1 and T2 (6 candidates per round)
  • 2017: Presidential election T1 and T2 (11 candidates per round)
  • 2022: Presidential election T1 and T2 (12 candidates per round)

Total Coverage:
  • Years: 3 (2012, 2017, 2022)
  • Rounds: 6 (T1 and T2 for each year)
  • Communes: {len(communes_in_fact)} (Rhône department)
  • Total result rows: {len(fact_resultats):,}

🔧 TECHNICAL IMPROVEMENTS:

2012 Data Harmonization:
  ✓ Added 'blancs_et_nuls' column for consistent historical comparison
  ✓ Maintains 'blancs' and 'nuls' separately for analytical granularity
  ✓ Supports unified metrics across all election years

Gold Layer Features:
  ✓ fact_participation includes: blancs_et_nuls, blancs_et_nuls_pct
  ✓ Harmonized column names (lowercase, underscores)
  ✓ Consistent schema across all {len(election_summary)} elections
  
📝 NOTES:

- Binary target remains: Macron wins 2022 T2 (macro: 1/0)
- Feature engineering can now leverage 10+ years of historical election data
- Security and demography indicators aligned with 2022 baseline
- Historical trends from 2012-2017 can improve prediction patterns

✨ Ready for advanced feature engineering:
  • Trend analysis (2012 → 2017 → 2022)
  • Swing voting patterns by nuance/party
  • Participation trend impact on outcomes
  • Long-term geographic voting patterns
""")

print("="*80 + "\n")

In [ ]:
# ============================================================================
# 📊 CORRECTED VISUALIZATION SUITE - WITH ACTUAL COLUMN NAMES
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from collections import defaultdict

# Configuration
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
sns.set_palette("husl")

print("✅ Visualization libraries loaded\n")

# ============================================================================
# HELPER: Convert numeric columns
# ============================================================================

def convert_numeric_columns(df, columns):
    """Convert specified columns to numeric type, handling errors."""
    df_copy = df.copy()
    for col in columns:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
    return df_copy

# ============================================================================
# 1️⃣ PARTICIPATION BY YEAR & ROUND
# ============================================================================

print("📊 Generating visualization 1: Participation Trends...")

# Convert fact_participation list to DataFrame and convert numeric columns
fact_part_df = pd.DataFrame(fact_participation)
numeric_cols = ['inscrits', 'votants', 'abstentions', 'blancs', 'nuls', 'blancs_et_nuls', 'exprimes']
fact_part_df = convert_numeric_columns(fact_part_df, numeric_cols)

# Aggregate by election
participation_summary = fact_part_df.groupby('id_election').agg({
    'inscrits': 'sum',
    'votants': 'sum',
    'abstentions': 'sum',
    'blancs': 'sum',
    'nuls': 'sum',
    'exprimes': 'sum'
}).reset_index()

# Calculate participation rates
participation_summary['pct_participation'] = (
    participation_summary['votants'] / participation_summary['inscrits'] * 100
)
participation_summary['pct_abstention'] = (
    participation_summary['abstentions'] / participation_summary['inscrits'] * 100
)
participation_summary['pct_blancs'] = (
    participation_summary['blancs'] / participation_summary['inscrits'] * 100
)
participation_summary['pct_nuls'] = (
    participation_summary['nuls'] / participation_summary['inscrits'] * 100
)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('📊 Participation Trends (2012-2022)', fontsize=16, fontweight='bold')

# 1. Participation rate by election
ax1 = axes[0, 0]
colors = ['#1f77b4' if 'T1' in e else '#ff7f0e' for e in participation_summary['id_election']]
participation_summary.plot(x='id_election', y='pct_participation', kind='bar', ax=ax1, 
                           color=colors, legend=False)
ax1.set_title('Voter Participation % by Election', fontweight='bold')
ax1.set_ylabel('Participation (%)')
ax1.set_xlabel('Election')
ax1.set_ylim([0, 100])
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# 2. Abstention components
ax2 = axes[0, 1]
abstention_data = participation_summary[['id_election', 'pct_abstention', 'pct_blancs', 'pct_nuls']]
abstention_data.set_index('id_election')[['pct_abstention', 'pct_blancs', 'pct_nuls']].plot(
    kind='bar', ax=ax2, color=['#d62728', '#2ca02c', '#9467bd'], width=0.8
)
ax2.set_title('Non-Votes Breakdown (Abstention, Blancs, Nuls)', fontweight='bold')
ax2.set_ylabel('% of Registered Voters')
ax2.set_xlabel('Election')
ax2.legend(['Abstention', 'Blancs', 'Nuls'], loc='upper right')
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Absolute voter numbers
ax3 = axes[1, 0]
participation_summary.set_index('id_election')[['inscrits', 'votants', 'abstentions']].plot(
    kind='bar', ax=ax3, width=0.8
)
ax3.set_title('Registered vs Actual Voters', fontweight='bold')
ax3.set_ylabel('Number of Voters')
ax3.set_xlabel('Election')
ax3.legend(loc='upper right')
ax3.grid(axis='y', alpha=0.3)
ax3.tick_params(axis='x', rotation=45)

# 4. Vote composition
ax4 = axes[1, 1]
vote_comp = participation_summary[['id_election', 'exprimes', 'blancs', 'nuls']].set_index('id_election')
vote_comp.plot(kind='bar', ax=ax4, stacked=True, color=['#1f77b4', '#2ca02c', '#9467bd'], width=0.8)
ax4.set_title('Valid Votes vs Non-Valid Votes (Stacked)', fontweight='bold')
ax4.set_ylabel('Number of Votes')
ax4.set_xlabel('Election')
ax4.legend(['Expressed', 'Blancs', 'Nuls'], loc='upper right')
ax4.grid(axis='y', alpha=0.3)
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print("✅ Visualization 1 complete\n")

# ============================================================================
# 2️⃣ NUANCE DISTRIBUTION & EVOLUTION
# ============================================================================

print("📊 Generating visualization 2: Nuance Distribution...")

# Convert fact_resultats to DataFrame
fact_res_df = pd.DataFrame(fact_resultats)
fact_res_df = convert_numeric_columns(fact_res_df, ['voix', 'pct_voix_exprimes'])

# Get nuance names from dim_nuance (using libelle_nuance, not nuance)
nuance_map = {str(n['id_nuance']): n['libelle_nuance'] for n in dim_nuance}
nuance_map[''] = 'Unknown'  # Handle empty nuance_id values

# Aggregate votes by nuance and election
nuance_agg = fact_res_df.groupby(['id_election', 'id_nuance']).agg({
    'voix': 'sum',
    'pct_voix_exprimes': 'mean'
}).reset_index()

nuance_agg['nuance_label'] = nuance_agg['id_nuance'].astype(str).fillna('').map(nuance_map)

# Pivot for visualization
nuance_pivot = nuance_agg.pivot_table(
    index='id_election', 
    columns='nuance_label', 
    values='voix', 
    aggfunc='sum'
).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🗳️  Political Nuance Distribution (2012-2022)', fontsize=16, fontweight='bold')

# Stacked bar chart
ax1 = axes[0]
nuance_pivot.plot(kind='bar', ax=ax1, stacked=False, width=0.8)
ax1.set_title('Vote Count by Political Nuance', fontweight='bold')
ax1.set_ylabel('Total Votes')
ax1.set_xlabel('Election')
ax1.legend(title='Nuance', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Percentage stacked bar
ax2 = axes[1]
nuance_pct = nuance_pivot.div(nuance_pivot.sum(axis=1), axis=0) * 100
nuance_pct.plot(kind='bar', ax=ax2, stacked=True, width=0.8)
ax2.set_title('Vote Share by Political Nuance (%)', fontweight='bold')
ax2.set_ylabel('Percentage (%)')
ax2.set_xlabel('Election')
ax2.legend(title='Nuance', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print("✅ Visualization 2 complete\n")

# ============================================================================
# 3️⃣ MACRON EVOLUTION (REM/LREM)
# ============================================================================

print("📊 Generating visualization 3: Macron Evolution...")

# Filter for Macron's party (LREM/REM/Renaissance)
macron_keywords = ['REM', 'LREM', 'Renaissance', 'Ensemble', 'République En Marche']
macron_ids = [
    k for k, v in nuance_map.items() 
    if any(kw.lower() in str(v).lower() for kw in macron_keywords)
]

if macron_ids:
    macron_data = fact_res_df[fact_res_df['id_nuance'].astype(str).isin(macron_ids)].copy()
    
    macron_by_election = macron_data.groupby('id_election').agg({
        'voix': 'sum'
    }).reset_index()
    
    # Merge with participation for percentage
    macron_by_election = macron_by_election.merge(
        participation_summary[['id_election', 'exprimes']], 
        on='id_election', 
        how='left'
    )
    macron_by_election['pct_expressed'] = (
        macron_by_election['voix'] / macron_by_election['exprimes'] * 100
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('📈 Macron/REM Vote Evolution (2017-2022)', fontsize=16, fontweight='bold')
    
    # Absolute votes
    ax1 = axes[0]
    macron_by_election.plot(x='id_election', y='voix', kind='bar', ax=ax1, 
                           color='skyblue', legend=False)
    ax1.set_title('Total Votes for Macron/REM', fontweight='bold')
    ax1.set_ylabel('Number of Votes')
    ax1.set_xlabel('Election')
    ax1.grid(axis='y', alpha=0.3)
    ax1.tick_params(axis='x', rotation=45)
    
    # Percentage of expressed votes
    ax2 = axes[1]
    macron_by_election.plot(x='id_election', y='pct_expressed', kind='bar', ax=ax2,
                           color='lightcoral', legend=False)
    ax2.set_title('Macron/REM % of Expressed Votes', fontweight='bold')
    ax2.set_ylabel('Percentage (%)')
    ax2.set_xlabel('Election')
    ax2.set_ylim([0, max(macron_by_election['pct_expressed'].fillna(0)) * 1.1])
    ax2.grid(axis='y', alpha=0.3)
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    print("✅ Visualization 3 complete\n")
else:
    print("⚠️ Macron/REM nuances not found in data\n")

# ============================================================================
# 4️⃣ SECURITY INDICATORS
# ============================================================================

print("📊 Generating visualization 4: Security Indicators...")

# Convert fact_securite to DataFrame
fact_sec_df = pd.DataFrame(fact_securite)
fact_sec_df = convert_numeric_columns(fact_sec_df, ['nombre', 'taux_pour_mille'])

# Get indicator names
indicator_map = {str(i['id_indicateur_securite']): i['indicateur'] for i in dim_indicateur}

# Filter for 2022 data and aggregate
fact_sec_2022 = fact_sec_df[fact_sec_df['annee'] == '2022'].copy()
fact_sec_2022['indicateur_name'] = fact_sec_2022['id_indicateur_securite'].astype(str).map(indicator_map)

sec_agg = fact_sec_2022.groupby('indicateur_name').agg({
    'nombre': 'sum',
    'taux_pour_mille': 'mean'
}).reset_index().sort_values('nombre', ascending=True)

# Keep top 10 indicators
sec_top = sec_agg.tail(10)

if len(sec_top) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('🛡️ Top Security Indicators in Rhône (2022)', fontsize=16, fontweight='bold')
    
    # Count of incidents
    ax1 = axes[0]
    ax1.barh(sec_top['indicateur_name'], sec_top['nombre'], color='coral')
    ax1.set_title('Total Incidents by Type', fontweight='bold')
    ax1.set_xlabel('Number of Incidents')
    ax1.grid(axis='x', alpha=0.3)
    
    # Rate per 1000 inhabitants
    ax2 = axes[1]
    ax2.barh(sec_top['indicateur_name'], sec_top['taux_pour_mille'], color='steelblue')
    ax2.set_title('Average Rate per 1000 Inhabitants', fontweight='bold')
    ax2.set_xlabel('Rate (per 1000)')
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print("✅ Visualization 4 complete\n")
else:
    print("⚠️ No security data available for 2022\n")

# ============================================================================
# 5️⃣ FEATURE IMPORTANCE FROM RANDOM FOREST MODEL
# ============================================================================

print("📊 Generating visualization 5: Feature Importance...")

if 'rf_model' in dir() and hasattr(rf_model, 'feature_importances_'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('🤖 Random Forest Feature Importance', fontsize=16, fontweight='bold')
    
    # Top 15 features
    ax1 = axes[0]
    if 'imp_df' in dir() and not imp_df.empty:
        imp_df.head(15).plot(kind='barh', ax=ax1, color='steelblue', legend=False)
        ax1.set_title('Top 15 Most Important Features', fontweight='bold')
        ax1.set_xlabel('Importance Score')
    else:
        top_features = sorted(enumerate(rf_model.feature_importances_), 
                            key=lambda x: x[1], reverse=True)[:15]
        indices, scores = zip(*top_features)
        ax1.barh(range(len(indices)), scores, color='steelblue')
        ax1.set_yticks(range(len(indices)))
        ax1.set_yticklabels([f'Feature {i}' for i in indices])
        ax1.set_title('Top 15 Most Important Features', fontweight='bold')
        ax1.set_xlabel('Importance Score')
    
    ax1.grid(axis='x', alpha=0.3)
    
    # Cumulative importance
    ax2 = axes[1]
    if 'imp_df' in dir() and not imp_df.empty:
        cum_importance = imp_df.cumsum()
        ax2.plot(range(len(cum_importance)), cum_importance.values, marker='o', 
                linewidth=2, markersize=6, color='green')
        ax2.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
        ax2.set_xlabel('Number of Features')
        ax2.set_ylabel('Cumulative Importance')
    else:
        cum_imp = np.cumsum(sorted(rf_model.feature_importances_, reverse=True))
        ax2.plot(range(len(cum_imp)), cum_imp, marker='o', linewidth=2, markersize=6, color='green')
        ax2.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
        ax2.set_xlabel('Number of Features')
        ax2.set_ylabel('Cumulative Importance')
    
    ax2.set_title('Cumulative Feature Importance', fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print("✅ Visualization 5 complete\n")
else:
    print("⚠️ Random Forest model not available\n")

# ============================================================================
# 6️⃣ MODEL PERFORMANCE COMPARISON
# ============================================================================

print("📊 Generating visualization 6: Model Performance Comparison...")

if 'comparison' in dir() and comparison:
    comparison_df = pd.DataFrame(comparison).T
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('🤖 Model Performance Comparison', fontsize=16, fontweight='bold')
    
    # Accuracy comparison
    ax1 = axes[0]
    if 'accuracy' in comparison_df.columns:
        acc_df = pd.to_numeric(comparison_df['accuracy'], errors='coerce')
        acc_df.plot(kind='barh', ax=ax1, color='lightgreen', legend=False)
        ax1.set_title('Model Accuracy', fontweight='bold')
        ax1.set_xlabel('Accuracy Score')
        ax1.set_xlim([0, 1])
        ax1.grid(axis='x', alpha=0.3)
    
    # F1 Score comparison
    ax2 = axes[1]
    if 'f1' in comparison_df.columns:
        f1_df = pd.to_numeric(comparison_df['f1'], errors='coerce')
        f1_df.plot(kind='barh', ax=ax2, color='lightblue', legend=False)
        ax2.set_title('F1 Score', fontweight='bold')
        ax2.set_xlabel('F1 Score')
        ax2.set_xlim([0, 1])
        ax2.grid(axis='x', alpha=0.3)
    else:
        print("⚠️ F1 scores not available in comparison data")
    
    plt.tight_layout()
    plt.show()
    print("✅ Visualization 6 complete\n")
else:
    print("⚠️ Comparison data not available\n")

print("\n🎉 All visualizations complete!")

In [ ]:
# ============================================================================
# 5️⃣ FEATURE IMPORTANCE FROM RANDOM FOREST MODEL
# ============================================================================

print("📊 Generating visualization 5: Feature Importance...")

if 'rf_model' in dir() and hasattr(rf_model, 'feature_importances_'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('🤖 Random Forest Feature Importance', fontsize=16, fontweight='bold')
    
    # Top 15 features
    ax1 = axes[0]
    if 'imp_df' in dir() and imp_df is not None and not imp_df.empty:
        # imp_df has columns ['Variable', 'Importance']
        top_15 = imp_df.head(15)
        
        # Create bar plot with explicit x and y
        ax1.barh(range(len(top_15)), top_15['Importance'].values, color='steelblue')
        ax1.set_yticks(range(len(top_15)))
        ax1.set_yticklabels(top_15['Variable'].values)
        ax1.set_title('Top 15 Most Important Features', fontweight='bold')
        ax1.set_xlabel('Importance Score')
        ax1.invert_yaxis()  # Highest at top
    else:
        # Fallback: use raw model importances
        top_features = sorted(enumerate(rf_model.feature_importances_), 
                            key=lambda x: x[1], reverse=True)[:15]
        indices, scores = zip(*top_features)
        ax1.barh(range(len(indices)), scores, color='steelblue')
        ax1.set_yticks(range(len(indices)))
        ax1.set_yticklabels([f'Feature {i}' for i in indices])
        ax1.set_title('Top 15 Most Important Features', fontweight='bold')
        ax1.set_xlabel('Importance Score')
        ax1.invert_yaxis()
    
    ax1.grid(axis='x', alpha=0.3)
    
    # Cumulative importance
    ax2 = axes[1]
    if 'imp_df' in dir() and imp_df is not None and not imp_df.empty:
        # Calculate cumulative sum from the importance values
        importances_values = imp_df['Importance'].values
        cum_importance = np.cumsum(importances_values)
        
        # Plot using explicit arrays
        ax2.plot(range(len(cum_importance)), cum_importance, marker='o', 
                linewidth=2, markersize=6, color='green')
        ax2.axhline(y=0.95 * cum_importance[-1], color='r', linestyle='--', 
                   label='95% threshold', linewidth=2)
        ax2.set_xlabel('Number of Features')
        ax2.set_ylabel('Cumulative Importance')
    else:
        # Fallback: use raw model importances
        sorted_importances = sorted(rf_model.feature_importances_, reverse=True)
        cum_imp = np.cumsum(sorted_importances)
        ax2.plot(range(len(cum_imp)), cum_imp, marker='o', linewidth=2, markersize=6, color='green')
        ax2.axhline(y=0.95 * cum_imp[-1], color='r', linestyle='--', 
                   label='95% threshold', linewidth=2)
        ax2.set_xlabel('Number of Features')
        ax2.set_ylabel('Cumulative Importance')
    
    ax2.set_title('Cumulative Feature Importance', fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print("✅ Visualization 5 complete\n")
else:
    print("⚠️ Random Forest model not available\n")

# ============================================================================
# 6️⃣ MODEL PERFORMANCE COMPARISON
# ============================================================================

print("📊 Generating visualization 6: Model Performance Comparison...")

if 'comparison' in dir() and comparison:
    comparison_df = pd.DataFrame(comparison).T
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('🤖 Model Performance Comparison', fontsize=16, fontweight='bold')
    
    # Accuracy comparison
    ax1 = axes[0]
    if 'accuracy' in comparison_df.columns:
        acc_df = pd.to_numeric(comparison_df['accuracy'], errors='coerce')
        acc_df.plot(kind='barh', ax=ax1, color='lightgreen', legend=False)
        ax1.set_title('Model Accuracy', fontweight='bold')
        ax1.set_xlabel('Accuracy Score')
        ax1.set_xlim([0, 1])
        ax1.grid(axis='x', alpha=0.3)
    
    # F1 Score comparison
    ax2 = axes[1]
    if 'f1' in comparison_df.columns:
        f1_df = pd.to_numeric(comparison_df['f1'], errors='coerce')
        f1_df.plot(kind='barh', ax=ax2, color='lightblue', legend=False)
        ax2.set_title('F1 Score', fontweight='bold')
        ax2.set_xlabel('F1 Score')
        ax2.set_xlim([0, 1])
        ax2.grid(axis='x', alpha=0.3)
    else:
        print("⚠️ F1 scores not available in comparison data")
    
    plt.tight_layout()
    plt.show()
    print("✅ Visualization 6 complete\n")
else:
    print("⚠️ Comparison data not available\n")

print("\n🎉 All visualizations complete!")